# Windows Repository Bridge — direct Colab vision server

The desktop app sends screenshots directly to this temporary authenticated HTTPS endpoint. Colab returns analysis to the app, which publishes the result through the normal GitHub mailbox tool result. The notebook does not read or write GitHub and **does not need a GitHub token or Colab Secret**.

1. Open this notebook from the Windows app's **Connect Colab** control while the app is running.
2. Connect a Colab runtime and run the worker cell. A GPU is needed only when you later enable the model, not to test basic connection reachability.
3. The cell checks the lightweight authenticated API, starts a temporary Cloudflare HTTPS tunnel, and performs a one-time localhost browser handshake. The Windows app then verifies the reverse path with an authenticated POST through the tunnel; the notebook waits for that server-to-server acknowledgment, not a browser-to-kernel callback. Failed connections stop with a stage-specific error.
4. After the app reports the service connected, use `computer(action=status)` to verify reachability. Use `computer(action=enable)` only when visual analysis is needed; the first activation may download about 6.2 GB and compile llama.cpp. Google Drive caches weights between runtimes.

If the output reports **no registration challenge**, rerun the cell in the tab newly opened by **Connect Colab**. If it reports `Failed to fetch` or `AbortError`, check Chrome's site settings for `colab.research.google.com` and allow Local Network/Loopback access if prompted or blocked; then click **Connect Colab** again for a fresh challenge. This connection failure stage is separate from GPU/model setup.

The endpoint requires a random in-memory bearer key, exposes only health/activation/deactivation/screenshot-analysis routes, and checks each screenshot SHA-256. The model/API is bound to localhost inside Colab; only the authenticated endpoint is tunneled. Deactivation leaves the Colab runtime and model running. Colab GPU access and session uptime are not guaranteed.

In [ ]:
# @title Start the direct Colab vision server
import base64
import hashlib
import hmac
import importlib.util
import json
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import requests
from google.colab import drive, output

MODEL_REPO = "Qwen/Qwen3-VL-8B-Instruct-GGUF"
MODEL_FILE = "Qwen3VL-8B-Instruct-Q4_K_M.gguf"
PROJECTOR_FILE = "mmproj-Qwen3VL-8B-Instruct-F16.gguf"
MODEL_URL = "https://huggingface.co/" + MODEL_REPO + "/resolve/main/" + MODEL_FILE
PROJECTOR_URL = "https://huggingface.co/" + MODEL_REPO + "/resolve/main/" + PROJECTOR_FILE
SERVER_URL = "http://127.0.0.1:8080"
DIRECT_API_PORT = 8765
DIRECT_PROTOCOL = "chatgpt-colab-vision-direct/v1"
RESULT_PROTOCOL = "chatgpt-colab-vision-result/v1"
MAX_SCREENSHOT_BYTES = 600 * 1024
POLL_SECONDS = 3
HEARTBEAT_SECONDS = 15
API_KEY = secrets.token_urlsafe(32)

# Make rerunning the cell replace the old tunnel/API threads, but keep a live loaded model.
_previous_api_server = globals().get("_direct_api_server")
_previous_tunnel_process = globals().get("_tunnel_process")
_previous_tunnel_log_handle = globals().get("_tunnel_log_handle")
_previous_model_server = globals().get("_server")
_previous_model_loaded = globals().get("_model_loaded", False)
_worker_generation = globals().get("_worker_generation", 0) + 1
if _previous_api_server is not None:
    try:
        _previous_api_server.shutdown()
        _previous_api_server.server_close()
    except Exception:
        pass
if _previous_tunnel_process is not None and _previous_tunnel_process.poll() is None:
    try:
        _previous_tunnel_process.terminate()
        _previous_tunnel_process.wait(timeout=8)
    except Exception:
        try:
            _previous_tunnel_process.kill()
        except Exception:
            pass
if _previous_tunnel_log_handle is not None:
    try:
        _previous_tunnel_log_handle.close()
    except Exception:
        pass
_server = _previous_model_server if _previous_model_server is not None and _previous_model_server.poll() is None else None
_model_loaded = bool(_server is not None and _previous_model_loaded)
_direct_api_server = None
_tunnel_process = None
_tunnel_log_handle = None
_enabled = False
_gpu = None
_vram_mib = None
_active_request = None
_last_completed = None
_runtime_lock = threading.RLock()
_inference_lock = threading.Lock()
_last_setup_attempt = 0.0

# The desktop confirms registration through an authenticated POST over the public tunnel.
# This server-to-server acknowledgement gates successful startup; browser JavaScript only initiates the one-time handshake.
_desktop_registration_event = threading.Event()
_desktop_registration_at = None

_stage_started_monotonic = time.monotonic()
_stage_started_at = datetime.now(timezone.utc).isoformat()
_last_stage_notice = None

_runtime_state = {
    "status": "inactive",
    "stage": "server_start",
    "message": "Direct Colab service is starting. Vision remains disabled until the app calls computer(action=enable).",
    "model_loaded": False,
    "gpu": None,
    "vram_mib": None,
    "active_request_id": None,
    "last_completed_request_id": None,
    "updated_at": None
}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def publish_runtime(status, stage=None, message=None, **extra):
    global _runtime_state, _stage_started_monotonic, _stage_started_at, _last_stage_notice
    with _runtime_lock:
        payload = dict(_runtime_state)
        next_stage = stage if stage is not None else payload.get("stage")
        next_message = (message if message is not None else payload.get("message") or "")[:1000]
        if next_stage != payload.get("stage"):
            _stage_started_monotonic = time.monotonic()
            _stage_started_at = utc_now()
        notice = (str(status), str(next_stage), next_message)
        if notice != _last_stage_notice:
            print("[vision] stage=" + str(next_stage) + " status=" + str(status) +
                  " elapsed=0s — " + next_message, flush=True)
            _last_stage_notice = notice
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "status": status,
            "enabled": bool(_enabled),
            "heartbeat_at": utc_now(),
            "updated_at": utc_now(),
            "worker": "colab-direct-vision/v1",
            "model_repository": MODEL_REPO,
            "model": MODEL_FILE,
            "projector": PROJECTOR_FILE,
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "stage": next_stage,
            "stage_started_at": _stage_started_at,
            "stage_elapsed_seconds": max(0, int(time.monotonic() - _stage_started_monotonic)),
            "message": next_message
        })
        payload.update(extra)
        _runtime_state = payload
        return dict(payload)

def check_gpu():
    if not shutil.which("nvidia-smi"):
        return None
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        check=True, capture_output=True, text=True, timeout=15
    )
    line = next((line.strip() for line in result.stdout.splitlines() if line.strip()), "")
    columns = [column.strip() for column in line.split(",")]
    if len(columns) < 3:
        raise RuntimeError("Could not parse nvidia-smi GPU/VRAM output.")
    return columns[0], int(columns[1]), int(columns[2])

def ensure_server():
    global _server, _model_loaded, _gpu, _vram_mib, _last_setup_attempt
    if _server is not None and _server.poll() is None:
        try:
            if requests.get(SERVER_URL + "/health", timeout=2).ok:
                _model_loaded = True
                publish_runtime("ready", stage="ready", message="The vision model is already loaded.")
                return True
        except requests.RequestException:
            pass
    if time.monotonic() - _last_setup_attempt < 45:
        return False
    _last_setup_attempt = time.monotonic()
    publish_runtime("starting", stage="drive_mount", message="Preparing persistent model storage. Google Drive authorization may be requested once.")
    try:
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        publish_runtime("needs_authentication", stage="drive_mount", message="Google Drive authorization is required in the Colab browser: " + str(exc))
        return False

    gpu = check_gpu()
    if not gpu:
        publish_runtime("gpu_unavailable", stage="detect_gpu", message="No NVIDIA GPU is attached to this Colab runtime. Select a GPU runtime if one is available.")
        return False
    _gpu, total_mib, free_mib = gpu
    _vram_mib = total_mib
    if total_mib < 10000:
        publish_runtime("gpu_unavailable", stage="check_vram", message=f"Detected {_gpu} with {total_mib} MiB VRAM. This 8B model and projector may not fit.")
        return False
    if not shutil.which("nvcc"):
        publish_runtime("error", stage="check_cuda", message="An NVIDIA GPU is present but nvcc is missing; the CUDA llama.cpp build cannot be verified.")
        return False

    root = Path("/content/drive/MyDrive/WindowsRepositoryBridgeVision")
    model_dir = root / "models"
    hf_cache = root / "hf-cache"
    model_dir.mkdir(parents=True, exist_ok=True)
    hf_cache.mkdir(parents=True, exist_ok=True)
    needed = [model_dir / MODEL_FILE, model_dir / PROJECTOR_FILE]
    free_bytes = shutil.disk_usage(root).free
    if any(not item.exists() for item in needed) and free_bytes < 7_000_000_000:
        publish_runtime("error", stage="check_drive_space", message=f"Persistent Drive storage has only {free_bytes // (1024**3)} GiB free; at least 7 GB is required.")
        return False

    publish_runtime("starting", stage="dependencies", message="Checking one-time model and build dependencies.")
    if importlib.util.find_spec("huggingface_hub") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True, timeout=300)
    from huggingface_hub import hf_hub_download

    publish_runtime("starting", stage="download_model", message="Reusing or downloading the Q4_K_M model and matching F16 vision projector.")
    model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir=str(model_dir), cache_dir=str(hf_cache))
    projector_path = hf_hub_download(repo_id=MODEL_REPO, filename=PROJECTOR_FILE, local_dir=str(model_dir), cache_dir=str(hf_cache))
    if Path(model_path).stat().st_size < 4_500_000_000 or Path(projector_path).stat().st_size < 1_000_000_000:
        raise RuntimeError("Model or projector file is unexpectedly small; refusing to use a partial download.")

    llama_source = Path("/content/llama.cpp")
    if not (llama_source / "CMakeLists.txt").exists():
        publish_runtime("starting", stage="clone_llama_cpp", message="Fetching upstream llama.cpp source.")
        subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp.git", str(llama_source)],
                       check=True, timeout=600)
    build_dir = llama_source / "build"
    server_binary = build_dir / "bin" / "llama-server"
    if not server_binary.exists():
        if not shutil.which("cmake") or not shutil.which("make"):
            subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=300)
            subprocess.run(["apt-get", "install", "-y", "-qq", "build-essential", "cmake", "git"],
                           check=True, timeout=600)
        publish_runtime("starting", stage="build_llama_cpp", message="Building llama-server with CUDA and multimodal support.")
        subprocess.run(["cmake", "-S", str(llama_source), "-B", str(build_dir),
                        "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF"],
                       check=True, timeout=900)
        subprocess.run(["cmake", "--build", str(build_dir), "--config", "Release",
                        "--target", "llama-server", "-j", "2"],
                       check=True, timeout=1800)
    if not server_binary.exists():
        raise RuntimeError("The llama.cpp build ended without a llama-server executable.")
    help_text = subprocess.run([str(server_binary), "--help"], check=True, capture_output=True, text=True, timeout=30).stdout
    for required_flag in ("--mmproj", "--host", "--port", "--ctx-size", "--n-gpu-layers"):
        if required_flag not in help_text:
            raise RuntimeError("The built llama-server does not expose required CLI flag: " + required_flag)

    if _server is not None and _server.poll() is None:
        _server.terminate()
        try:
            _server.wait(timeout=10)
        except subprocess.TimeoutExpired:
            _server.kill()
    log_path = Path("/content/llama-server-vision.log")
    log_handle = open(log_path, "a", encoding="utf-8")
    _server = subprocess.Popen([
        str(server_binary), "-m", str(model_path), "--mmproj", str(projector_path),
        "-c", "8192", "--n-gpu-layers", "99", "--host", "127.0.0.1", "--port", "8080"
    ], stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    publish_runtime("starting", stage="load_model", message="llama-server started; waiting for model health.")
    deadline = time.monotonic() + 240
    while time.monotonic() < deadline:
        if _server.poll() is not None:
            try:
                tail = log_path.read_text(encoding="utf-8", errors="replace")[-2500:]
            except Exception:
                tail = "(server log unavailable)"
            raise RuntimeError("llama-server exited before reporting healthy. Diagnostic tail: " + tail)
        try:
            if requests.get(SERVER_URL + "/health", timeout=3).ok:
                _model_loaded = True
                publish_runtime("ready", stage="ready", message="Model and vision projector are loaded and llama-server health succeeded.")
                log_handle.close()
                return True
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("Timed out waiting for llama-server /health after loading the Qwen3-VL model.")

class DirectVisionHandler(BaseHTTPRequestHandler):
    server_version = "BridgeVision/1"

    def log_message(self, fmt, *args):
        # Do not log bearer credentials, image payloads, or user prompts.
        return

    def _authorized(self):
        value = self.headers.get("Authorization", "")
        return hmac.compare_digest(value, "Bearer " + API_KEY)

    def _send(self, status, payload):
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Cache-Control", "no-store")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        if self.path != "/health":
            self._send(404, {"error": "Not found"})
            return
        with _runtime_lock:
            payload = dict(_runtime_state)
            payload["stage_started_at"] = _stage_started_at
            payload["stage_elapsed_seconds"] = max(0, int(time.monotonic() - _stage_started_monotonic))
        payload.update({
            "protocol": DIRECT_PROTOCOL,
            "enabled": bool(_enabled),
            "model_loaded": bool(_model_loaded),
            "gpu": _gpu,
            "vram_mib": _vram_mib,
            "active_request_id": _active_request,
            "last_completed_request_id": _last_completed,
            "heartbeat_at": utc_now(),
            "updated_at": utc_now()
        })
        self._send(200, payload)

    def do_POST(self):
        global _enabled, _active_request, _last_completed
        if not self._authorized():
            self._send(401, {"error": "Unauthorized"})
            return
        try:
            length = int(self.headers.get("Content-Length", "0"))
        except ValueError:
            self._send(400, {"error": "Invalid Content-Length"})
            return
        if length < 1 or length > 2_000_000:
            self._send(413, {"error": "Request body must be between 1 byte and 2 MB"})
            return
        try:
            body = json.loads(self.rfile.read(length).decode("utf-8"))
        except Exception:
            self._send(400, {"error": "Malformed JSON"})
            return
        if not isinstance(body, dict) or body.get("protocol") != DIRECT_PROTOCOL:
            self._send(400, {"error": "Protocol mismatch"})
            return
        if self.path == "/registration/ack":
            global _desktop_registration_at
            _desktop_registration_at = utc_now()
            publish_runtime(
                "starting",
                stage="desktop_registration_ack",
                message="The Windows app verified the authenticated HTTPS tunnel. Vision remains disabled and the model remains unloaded.",
                desktop_registered=True,
                desktop_registered_at=_desktop_registration_at
            )
            _desktop_registration_event.set()
            self._send(200, {
                "protocol": DIRECT_PROTOCOL,
                "accepted": True,
                "desktop_registered": True,
                "desktop_registered_at": _desktop_registration_at
            })
            return
        if self.path == "/activate":
            _enabled = True
            publish_runtime("starting", stage="activation", message="Vision activation received. Model setup is starting.")
            self._send(202, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": True})
            return
        if self.path == "/deactivate":
            _enabled = False
            publish_runtime("inactive", stage="idle", message="Vision is disabled; the runtime and any loaded model remain running.")
            self._send(200, {"protocol": DIRECT_PROTOCOL, "accepted": True, "enabled": False, "runtime_kept_alive": True})
            return
        if self.path != "/analyze":
            self._send(404, {"error": "Not found"})
            return
        if not _enabled:
            self._send(409, {"error": "Vision is inactive; call computer(action=enable) first."})
            return
        if not _model_loaded:
            self._send(409, {"error": "Vision model is not loaded yet; check computer(action=status) and retry the same request ID."})
            return
        with _inference_lock:
            try:
                job_id = body.get("request_id")
                if not isinstance(job_id, str) or not re.fullmatch(r"computer-[a-f0-9]{24}", job_id):
                    self._send(400, {"error": "Malformed request ID"})
                    return
                expected_hash = body.get("sha256")
                image_b64 = body.get("image_base64")
                if not isinstance(expected_hash, str) or not re.fullmatch(r"[a-f0-9]{64}", expected_hash):
                    self._send(400, {"error": "Malformed SHA-256"})
                    return
                if not isinstance(image_b64, str):
                    self._send(400, {"error": "Image payload is missing"})
                    return
                image_bytes = base64.b64decode(image_b64, validate=True)
                if not image_bytes or len(image_bytes) > MAX_SCREENSHOT_BYTES:
                    self._send(413, {"error": "Screenshot is empty or larger than the 600 KiB limit"})
                    return
                actual_hash = hashlib.sha256(image_bytes).hexdigest()
                if not hmac.compare_digest(actual_hash, expected_hash):
                    self._send(400, {"error": "Screenshot SHA-256 did not match the submitted value"})
                    return
                _active_request = job_id
                publish_runtime("processing", stage="inference", message="Analyzing one screenshot received from the desktop app.")
                started = time.monotonic()
                request_payload = {
                    "model": "qwen3-vl-8b-vision",
                    "temperature": 0.1,
                    "max_tokens": 1800,
                    "stream": False,
                    "messages": [
                        {"role": "system", "content": (
                            "You are a careful visual observer, not a general reasoning agent. "
                            "Describe only details supported by the provided image. Do not infer unseen state. "
                            "Mark illegible text and uncertainty explicitly. Organize the report into: Overview; "
                            "Visible objects, panels, controls and layout; Text transcription; Visible errors or anomalies; "
                            "Evidence supporting observations; Uncertainties."
                        )},
                        {"role": "user", "content": [
                            {"type": "text", "text": str(body.get("prompt") or "")[:2000]},
                            {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + image_b64}}
                        ]}
                    ]
                }
                try:
                    response = requests.post(SERVER_URL + "/v1/chat/completions", json=request_payload, timeout=(10, 300))
                    if not response.ok:
                        raise RuntimeError("llama-server HTTP " + str(response.status_code) + ": " + response.text[:600])
                    output = response.json()
                    choices = output.get("choices")
                    if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
                        raise RuntimeError("llama-server returned a malformed completion payload.")
                    message = choices[0].get("message", {})
                    report = message.get("content") if isinstance(message, dict) else None
                    if isinstance(report, list):
                        report = "\n".join(str(part.get("text", "")) for part in report if isinstance(part, dict))
                    if not isinstance(report, str) or not report.strip():
                        raise RuntimeError("The model returned an empty or non-text report.")
                    report = report.strip()
                    if len(report) > 30000:
                        report = report[:30000] + "\n\n[Report truncated at the safety limit.]"
                    _last_completed = job_id
                    result = {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "completed",
                        "completed_at": utc_now(),
                        "model_repository": MODEL_REPO,
                        "model": MODEL_FILE,
                        "input_sha256": actual_hash,
                        "inference_ms": int((time.monotonic() - started) * 1000),
                        "report": report
                    }
                    self._send(200, result)
                except Exception as exc:
                    self._send(200, {
                        "protocol": RESULT_PROTOCOL,
                        "request_id": job_id,
                        "status": "failed",
                        "completed_at": utc_now(),
                        "input_sha256": actual_hash,
                        "error": type(exc).__name__ + ": " + str(exc)[:1200]
                    })
                finally:
                    _active_request = None
                    publish_runtime("ready" if _model_loaded else "error", stage="ready" if _model_loaded else "inference_error",
                                    message="Model is ready for another request." if _model_loaded else "Model health was lost during inference.")
            except Exception as exc:
                self._send(400, {"error": type(exc).__name__ + ": " + str(exc)[:800]})

def api_server_loop():
    global _direct_api_server
    server = ThreadingHTTPServer(("127.0.0.1", DIRECT_API_PORT), DirectVisionHandler)
    server.daemon_threads = True
    _direct_api_server = server
    print("Authenticated vision API listening on localhost port", DIRECT_API_PORT)
    server.serve_forever()

def worker_loop(generation):
    while generation == _worker_generation:
        try:
            if _enabled:
                if not _model_loaded:
                    try:
                        ensure_server()
                    except Exception as exc:
                        publish_runtime("error", stage="setup_error", message=type(exc).__name__ + ": " + str(exc))
                else:
                    publish_runtime("ready", stage="ready", message="Vision is enabled and the model is ready.")
            else:
                publish_runtime("inactive", stage="idle", message="Direct Colab server is connected; vision is disabled and model setup will not start.")
        except Exception as exc:
            publish_runtime("error", stage="worker_error", message=type(exc).__name__ + ": " + str(exc))
        time.sleep(POLL_SECONDS)

def ensure_cloudflared():
    binary = shutil.which("cloudflared")
    if binary:
        print("[tunnel] Reusing cloudflared binary already on PATH.")
        return binary
    binary_path = Path("/content/cloudflared")
    if not binary_path.exists():
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        print("[tunnel] Downloading cloudflared binary (network timeout: 90 seconds).")
        started = time.monotonic()
        try:
            response = requests.get(url, timeout=90)
            response.raise_for_status()
        except Exception as exc:
            raise RuntimeError("cloudflared binary download failed after " +
                               str(round(time.monotonic() - started, 1)) + " seconds: " +
                               type(exc).__name__ + ": " + str(exc)[:500]) from exc
        binary_path.write_bytes(response.content)
        binary_path.chmod(0o755)
        print("[tunnel] cloudflared download completed in " +
              str(round(time.monotonic() - started, 1)) + " seconds.")
    if not os.access(binary_path, os.X_OK):
        binary_path.chmod(0o755)
    return str(binary_path)


def start_cloudflared():
    global _tunnel_process, _tunnel_log_handle
    binary = ensure_cloudflared()
    log_path = Path("/content/bridge-cloudflared.log")
    _tunnel_log_handle = open(log_path, "w", encoding="utf-8")
    tunnel_started = time.monotonic()
    _tunnel_process = subprocess.Popen(
        [binary, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:" + str(DIRECT_API_PORT)],
        stdout=_tunnel_log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
    print("[tunnel] cloudflared started; waiting up to 90 seconds for an HTTPS URL.")
    deadline = time.monotonic() + 90
    pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    while time.monotonic() < deadline:
        if _tunnel_process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-1600:]
            raise RuntimeError("cloudflared exited after " +
                               str(round(time.monotonic() - tunnel_started, 1)) +
                               " seconds before creating its HTTPS tunnel: " + tail)
        text = log_path.read_text(encoding="utf-8", errors="replace")
        match = pattern.search(text)
        if match:
            print("[tunnel] HTTPS URL created in " +
                  str(round(time.monotonic() - tunnel_started, 1)) + " seconds.")
            return match.group(0)
        time.sleep(1)
    raise RuntimeError("Timed out after " + str(round(time.monotonic() - tunnel_started, 1)) +
                       " seconds waiting for a Cloudflare Quick Tunnel URL. Check /content/bridge-cloudflared.log.")

def verify_local_api_health():
    """Verify the authenticated API without touching the GPU or loading the model."""
    url = "http://127.0.0.1:" + str(DIRECT_API_PORT) + "/health"
    deadline = time.monotonic() + 8
    last_error = "No response yet."
    while time.monotonic() < deadline:
        try:
            response = requests.get(url, headers={"Authorization": "Bearer " + API_KEY}, timeout=2)
            if response.status_code == 200:
                payload = response.json()
                if payload.get("protocol") == DIRECT_PROTOCOL and payload.get("enabled") is False and isinstance(payload.get("model_loaded"), bool):
                    return
                last_error = "Health payload protocol or enabled/model-readiness state was unexpected."
            else:
                last_error = "HTTP " + str(response.status_code)
        except Exception as exc:
            last_error = type(exc).__name__ + ": " + str(exc)[:300]
        time.sleep(0.4)
    raise RuntimeError("Local authenticated API health check failed before tunnel setup: " + last_error)


def stop_connection_bootstrap():
    """Close the API and tunnel on failure without terminating a previously cached model process."""
    global _direct_api_server, _tunnel_process, _tunnel_log_handle
    server, _direct_api_server = _direct_api_server, None
    if server is not None:
        try:
            server.shutdown()
        except Exception:
            pass
        try:
            server.server_close()
        except Exception:
            pass
    tunnel, _tunnel_process = _tunnel_process, None
    if tunnel is not None and tunnel.poll() is None:
        try:
            tunnel.terminate()
            tunnel.wait(timeout=5)
        except Exception:
            try:
                tunnel.kill()
                tunnel.wait(timeout=3)
            except Exception:
                pass
    handle, _tunnel_log_handle = _tunnel_log_handle, None
    if handle is not None:
        try:
            handle.close()
        except Exception:
            pass


def register_tunnel_with_desktop(tunnel_url):
    global _desktop_registration_at
    _desktop_registration_event.clear()
    _desktop_registration_at = None
    payload_json = json.dumps({
        "protocol": DIRECT_PROTOCOL,
        "tunnel_url": tunnel_url,
        "api_key": API_KEY
    })
    js = r"""
(async () => {
  const payload = __PAYLOAD__;
  const panelId = "windows-bridge-colab-registration";
  const oldPanel = document.getElementById(panelId);
  if (oldPanel) oldPanel.remove();
  const panel = document.createElement("section");
  panel.id = panelId;
  panel.setAttribute("role", "status");
  panel.style.cssText = "position:fixed;right:16px;bottom:16px;z-index:2147483647;width:340px;max-width:calc(100vw - 32px);box-sizing:border-box;padding:14px;background:#fff;color:#202124;border:1px solid #9aa0a6;border-radius:10px;box-shadow:0 4px 20px rgba(0,0,0,.24);font:14px/1.4 Arial,sans-serif;";
  const title = document.createElement("div");
  title.textContent = "Windows Repository Bridge";
  title.style.cssText = "font-weight:700;font-size:15px;margin-bottom:6px;";
  const detail = document.createElement("div");
  detail.style.cssText = "overflow-wrap:anywhere;margin-bottom:10px;";
  const button = document.createElement("button");
  button.type = "button";
  button.textContent = "Connect Windows Bridge";
  button.style.cssText = "padding:8px 12px;border:0;border-radius:6px;background:#1a73e8;color:#fff;font-weight:600;cursor:pointer;";
  panel.append(title, detail, button);
  document.body.appendChild(panel);

  const origin = String(window.location.origin || "unknown");
  const setStatus = (message, failed=false) => {
    detail.textContent = String(message || "").slice(0,1400);
    detail.style.color = failed ? "#b3261e" : "#202124";
  };
  let busy = false;
  let registered = false;
  const getPermission = async () => {
    try {
      if (!navigator.permissions || typeof navigator.permissions.query !== "function") return "prompt";
      return (await navigator.permissions.query({name:"loopback-network"})).state || "prompt";
    } catch (_) { return "prompt"; }
  };
  const connect = async permission => {
    if (busy || registered) return;
    if (permission === "denied") {
      setStatus("Chrome currently denies loopback access. Open the site controls beside the address bar, open Site settings, set Local network access to Allow, then reload the notebook and rerun this cell.", true);
      return;
    }
    if (origin !== "https://colab.research.google.com") {
      setStatus("This control is running in " + origin + ", not the top-level Colab page. Reopen the notebook in the normal Colab tab and rerun the cell.", true);
      return;
    }
    busy = true;
    button.disabled = true;
    button.style.opacity = ".65";
    setStatus("Connecting to the local Windows Bridge. If Chrome prompts for Local Network Access, choose Allow.");
    try {
      const deadline = Date.now() + 60000;
      let lastError = "No local Bridge listener responded.";
      let appPort = null;
      let nonce = null;
      for (let port = 17843; port <= 17854; port++) {
        if (Date.now() >= deadline) break;
        const controller = new AbortController();
        const timer = setTimeout(() => controller.abort(), port === 17843 ? 60000 : 3000);
        try {
          const response = await fetch("http://127.0.0.1:" + port + "/vision/challenge", {
            method:"GET", mode:"cors", cache:"no-store", credentials:"omit",
            targetAddressSpace:"loopback", signal:controller.signal
          });
          let challenge = {};
          try { challenge = await response.json(); } catch (_) {}
          if (response.ok && challenge.ok === true && challenge.protocol === payload.protocol &&
              typeof challenge.nonce === "string" && /^[a-f0-9]{64}$/.test(challenge.nonce)) {
            appPort = port;
            nonce = challenge.nonce;
            break;
          }
          lastError = String(challenge.error || ("Challenge endpoint returned HTTP " + response.status)).slice(0,500);
        } catch (e) { lastError = String(e && e.message || e).slice(0,500); }
        finally { clearTimeout(timer); }
      }
      if (appPort === null || nonce === null) {
        throw new Error("The Windows app's one-time challenge was unreachable. Browser origin: " + origin +
          "; loopback permission: " + permission + "; detail: " + lastError +
          ". Allow Local network access for colab.research.google.com if prompted, then retry.");
      }
      setStatus("Local access granted. Registering the HTTPS tunnel and verifying the return connection...");
      const controller = new AbortController();
      const timer = setTimeout(() => controller.abort(), 20000);
      try {
        const response = await fetch("http://127.0.0.1:" + appPort + "/vision/register", {
          method:"POST", mode:"cors", cache:"no-store", credentials:"omit",
          targetAddressSpace:"loopback", headers:{"Content-Type":"application/json"},
          body:JSON.stringify({...payload, nonce}), signal:controller.signal
        });
        let result = {};
        try { result = await response.json(); } catch (_) {}
        if (!response.ok || result.ok !== true || result.protocol !== payload.protocol || result.tunnel_verified !== true) {
          throw new Error(String(result.error || ("Registration returned HTTP " + response.status)).slice(0,900));
        }
        registered = true;
        button.textContent = "Connected";
        button.disabled = true;
        button.style.opacity = "1";
        setStatus("Connected. The Windows app verified the authenticated HTTPS tunnel. Vision remains disabled; no model was loaded.");
      } finally { clearTimeout(timer); }
    } catch (e) {
      setStatus("Connection failed: " + String(e && e.message || e).slice(0,1000), true);
    } finally {
      busy = false;
      if (!registered) {
        button.disabled = false;
        button.style.opacity = "1";
        button.textContent = "Retry Windows Bridge connection";
      }
    }
  };

  button.addEventListener("click", () => { void connect(permissionState); });
  const permissionState = await getPermission();
  if (permissionState === "granted") {
    setStatus("Local Network Access is already allowed. Checking the Windows Bridge...");
    void connect(permissionState);
  } else if (permissionState === "denied") {
    setStatus("Chrome denies loopback access. Open the site controls beside the address bar, open Site settings, set Local network access to Allow, then reload the notebook and rerun this cell.", true);
    button.textContent = "Retry after changing site settings";
  } else {
    setStatus("Click Connect Windows Bridge. If Chrome asks to allow Local Network Access, choose Allow. This permits the Colab page to reach the local Bridge listener; vision stays off and no model is loaded.");
  }
  return {installed:true, origin, loopbackPermission:permissionState};
})()
""".replace("__PAYLOAD__", payload_json)
    try:
        ui_state = output.eval_js(js)
    except Exception as exc:
        stop_connection_bootstrap()
        raise RuntimeError(
            "Could not create the top-level Colab connection control: " +
            type(exc).__name__ + ": " + str(exc)[:800] +
            ". No model was loaded. Reopen the notebook in the normal Colab page and rerun this cell."
        ) from exc
    print("[browser] Installed the top-level Colab connection control:", json.dumps(ui_state, ensure_ascii=False))
    print("[browser] If permission is not already granted, click 'Connect Windows Bridge' on the page and accept Chrome's Local Network Access prompt.")


print("[1/3] Starting the lightweight authenticated Colab API. The model remains unloaded.")
publish_runtime("starting", stage="api_server", message="Starting the lightweight authenticated vision API. The model remains unloaded.")
threading.Thread(target=api_server_loop, name="bridge-direct-api", daemon=True).start()
time.sleep(1)
try:
    verify_local_api_health()
except Exception:
    stop_connection_bootstrap()
    raise
print("[1/3] Local API reachability and bearer authentication passed.")

print("[2/3] Creating the temporary HTTPS tunnel (maximum 90 seconds).")
try:
    tunnel_url = start_cloudflared()
except Exception:
    stop_connection_bootstrap()
    raise
print("[2/3] Tunnel running. Sending the browser registration handshake; model remains unloaded.")
register_tunnel_with_desktop(tunnel_url)

print("[3/3] Waiting up to 80 seconds for the Windows app's authenticated server-to-server acknowledgement.")
if not _desktop_registration_event.wait(timeout=80):
    stop_connection_bootstrap()
    raise RuntimeError(
        "The Windows app did not confirm the authenticated HTTPS tunnel within 80 seconds. "
        "The cell installs a top-level Colab connection panel. If Chrome needs permission, click 'Connect Windows Bridge' "
        "in that panel and allow Local Network Access. Read the panel's visible error for the precise browser failure. "
        "This is a connection-stage failure, not a GPU/model-loading failure; the tunnel was stopped and no model was loaded."
    )

print("[3/3] Direct authenticated HTTPS connection registered successfully. Vision remains disabled.")
print("Use computer(action=status) to verify reachability. Use computer(action=enable) only when you want to load the model.")
print("The connection check did not download or activate the model. Keep this cell running while using vision.")
threading.Thread(target=worker_loop, args=(_worker_generation,), name="bridge-vision-worker", daemon=True).start()
while True:
    time.sleep(60)
